## Comprehension - Working with LLM APIs

### Import Libraries and Load Data

Import libraries, load the API keys, and setup the client

In [ ]:
# Install the LLM provider SDK/library of your choice (Gemini/Hugging Face/OpenAI) and load the API Key

# %pip install -q -U google-genai huggingface_hub python-dotenv openai

In [38]:
# Imports
import os
from dotenv import load_dotenv

from google import genai
from huggingface_hub import InferenceClient
from openai import OpenAI

In [39]:
# Define client
# Load API credentials securely
load_dotenv(override=True)

# Initialize the Gemini Client
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# --------------------------------------------------
# Clients
# --------------------------------------------------

# Gemini
gemini_client = genai.Client(
    api_key=GOOGLE_API_KEY
)

# Hugging Face
hf_client = InferenceClient(
    token=HF_TOKEN
)

# OpenAI
openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)

Load the dataset

In [40]:
# Load CSV into any type of data structure you prefer

import pandas as pd

# Load CSV
df = pd.read_csv("laptop_descriptions.csv")

# Inspect the data
print(df.head())
print(df.shape)
print(df.columns)
df.iloc[1]

                                  laptop_description
0  The Dell Inspiron is a versatile laptop that c...
1  The MSI GL65 is a high-performance laptop desi...
2  The HP EliteBook is a premium laptop designed ...
3  The Lenovo IdeaPad is a versatile laptop that ...
4  The ASUS ZenBook Pro is a high-end laptop that...
(20, 1)
Index(['laptop_description'], dtype='object')


laptop_description    The MSI GL65 is a high-performance laptop desi...
Name: 1, dtype: object

---

## Task 1

Based on the dataset, classify the laptop into one of the following tags corresponding to their categories:
- general
- business
- gamer
- programmer
- multimedia

| Category | Description |
| --------------- | --------------- |
| general | For general purpose use such as light web browsing, editing documents etc. |
| business | For business users, the focus is on portability, battery backup and general purpose use. |
| gamer | For gamers, the focus is primarily on high-performance, a separate GPU for high-performance graphics, a high-end CPU processor etc. |
| programmer | For programmers, the focus is on performance, battery backup, high-end RAM etc. |
| multimedia | For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc. |


The final output dataframe should like the table below:

| laptop_description | category |
|--------------- |--------------- |
| The Dell Inspiron ... | **general** |
| The MSI GL65 ... | **gaming** |
|  ... | **...** |


### <font color='RED' >Question 1</font>

What is the laptop category of the laptop at the first index position (i.e. MSI GL65)? Use the following code to obtain the result: 

`df.iloc[1]['Category']`

Attempt this question through the 3 tasks given below.

#### **1.1** First, try creating a wrapper function which takes in a user request, and outputs one word - the category name.

This function will involve creating a system message to define the role, context, task, and output style. Note that you have to ensure one word outputs. This function will produce the output only for one laptop description, not the whole data.

Additionally, try using error handling as well.


You do not need to provide the category-related information or laptop data here. That will be given with user prompt.

In [41]:
### Function to get LLM Response

def get_chat_response_q1(user_request, model="gemini"):

    system_message = """
    You are a laptop classification assistant.

    Your task is to identify the category of the laptop described by the user.

    Return ONLY the category name.
    Return exactly ONE word.
    Do not provide any explanation, punctuation, or additional text.
    """

    try:

        if model.lower() == "gemini":

            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash",
                contents=[
                    {
                        "role": "user",
                        "parts": [
                            {"text": system_message},
                            {"text": user_request}
                        ]
                    }
                ]
            )

            result = response.text.strip()

        elif model.lower() == "hf":

            response = hf_client.chat_completion(
                model="meta-llama/Llama-3.1-8B-Instruct",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": user_request}
                ],
                max_tokens=10,
                temperature=0
            )

            result = response.choices[0].message.content.strip()

        else:
            raise ValueError(
                f"Unsupported model: {model}. Use 'gemini' or 'hf'."
            )

        # Ensure exactly one word
        if len(result.split()) != 1:
            raise ValueError(
                f"Expected one-word response, but received: '{result}'"
            )

        return result

    except Exception as e:
        print(f"Error: {e}")
        return None

#### **1.2** Prompt Definition

Now define the user prompt where you will give the description of laptops and the description of each category (use category descriptions from the table above). Include the exact task to be performed over these.

You can use f-string to set up a prompt which has a placeholder for the dataset (description of laptops). In the next part, we will attach the dataset to this f-string based prompt.

Note that a good way to provide the information about a laptop to the user prompt will be to iterate over the whole data and attach it to the placeholder row-by-row. Design your prompt keeping this in mind.

In [42]:
# PROMPT DEFINITION

category_md = """

| Category | Description |
| --------------- | --------------- |
| general | For general purpose use such as light web browsing, editing documents etc. |
| business | For business users, the focus is on portability, battery backup and general purpose use. |
| gamer | For gamers, the focus is primarily on high-performance, a separate GPU for high-performance graphics, a high-end CPU processor etc. |
| programmer | For programmers, the focus is on performance, battery backup, high-end RAM etc. |
| multimedia | For multimedia use cases, the requirements are a good quality/ high resolution display, wide screens, good audio and video quality, battery backup, efficiency etc. |

"""

mcq1_prompt = '''
Here are the laptop categories and their descriptions in a markdown table formal:

{category_descriptions}

Classify the following laptop into one of the categories above:

{laptop_description}
'''



In [43]:
prompt = mcq1_prompt.format(
    category_descriptions=category_md,
    laptop_description=df.iloc[6]['laptop_description']
)

response = get_chat_response_q1(prompt, "hf")
print(response)

multimedia


#### **1.3** Tagging Laptops

Now, create a final wrapper function to combine the placeholder prompt with the dataset, and use the first wrapper function created in task 1.1 to assign a category to each laptop in the data. You can iterate over every laptop and assign it a category.

In [57]:
def tag_laptop(model="gemini"):

    laptop_df = df.copy()
    laptop_dict = laptop_df.to_dict(orient='records')

    for i in range(5):  #range(len(laptop_dict)):

        # print(f"[{i + 1}/{len(laptop_dict)}] Processing laptop...")
        print(f"[{i + 1}/{5} Processing laptop...")

        prompt = mcq1_prompt.format(
            category_descriptions=category_md,
            laptop_description=laptop_dict[i]['laptop_description']
        )

        print(f"  Description: {laptop_dict[i]['laptop_description'][:80]}...")

        laptop_category = get_chat_response_q1(prompt, model)

        print(f"  Category: {laptop_category}")

        laptop_df.at[i, 'Category'] = laptop_category

    return laptop_df

In [58]:
# Finding the category of a laptop at the asked index
tag_df = tag_laptop(model="hf")
tag_df.iloc[1]['Category']

[1/5 Processing laptop...
  Description: The Dell Inspiron is a versatile laptop that combines powerful performance and a...
  Category: general
[2/5 Processing laptop...
  Description: The MSI GL65 is a high-performance laptop designed for gaming enthusiasts. Power...
  Category: gamer
[3/5 Processing laptop...
  Description: The HP EliteBook is a premium laptop designed for professionals who value perfor...
  Category: business
[4/5 Processing laptop...
  Description: The Lenovo IdeaPad is a versatile laptop that balances performance and affordabi...
  Category: general
[5/5 Processing laptop...
  Description: The ASUS ZenBook Pro is a high-end laptop that offers exceptional performance an...
  Category: Gamer


'gamer'

**How many gamer laptops did you find?**

In [59]:
# Total number of products of each category
tag_df['Category'].value_counts()

Category
general     2
gamer       1
business    1
Gamer       1
Name: count, dtype: int64

---

## Task 2: Information Extraction


Extract the relevant values for the following dictionary items from the product description.
```
{
    "Brand": {"type": "string"},
    "Model Name": {"type": "string"},
    "GPU processor": {"type": "string"},
    "Display Resolution": {"type": "string"},
    "Weight": {"type": "string"},
    "Processor": {"type": "string"},
    "Clock speed": {"type": "string"},
    "Budget": {"type": "string"}
}
```

You can add this property dictionary for all the products to a list and output that list.

### <font color='RED' >Question 2</font>

Which of the following correctly represents the laptop tag values for the Laptop description at index position 10? The laptop is ASUS ROG Strix G.

Attempt this question through the 3 tasks below

---

Note: Most models like Gemini and GPT now use a Pydantic schema for structure validation. Pydantic is a library for data validation that enforces data structures, and coerces input data to predined data types.

The structure is usually defined using a `BaseModel` class. It has been already done for you below.

In [ ]:
from pydantic import BaseModel

class Laptop(BaseModel):
    Brand: str
    Model_Name: str
    GPU_processor: str
    Display_Resolution: str
    Weight: str
    Processor: str
    Clock_speed: str
    Budget: str

This created schema can directly be passed with the model call which returns a Pydantic object adhering to it.

In [ ]:
# Gemini
response = client.models.generate_content(
    model=,
    contents=,
    config={
        "response_mime_type": "application/json",
        "response_schema": Laptop
    }
)

# OpenAI (.parse instead of .create)
response = client.responses.parse(
    model=,
    input=,
    text_format=Laptop,
)
response.output_parsed

#### **2.1** Prompt Definition

First define the user prompt where you will give the structure of the output, define the exact task, and again add the placeholder for laptop description using f-string.

You may want to use a fixed schema for structured output for this task. Though, you can first try using a simple prompt.

Ensure the details extracted are quantitative, not generic (processing speed: "2.4 GHz" and not "low")

In [ ]:
# Define user prompt

mcq2_prompt = '''

'''

# We are giving the product descriptions as well as the dictionary structure of our required output

#### **2.2** Function for Response Generation

Now, define a function that takes the Laptop Pydantic schema and generates an output for each laptop. Define system instructions and response type as well.

Similar to task 1.1, this will also work on singular entries of laptops

In [ ]:
# Function to get response

def get_chat_response_q2(user_request):

  

#### **2.3** Extracting Data

Finally, create a final wrapper function to iterate over the laptop data like in task 1.3, provide the description in each row to the user query, and get the responses.

In [ ]:
# Write the code to extract product information from the 'Description' value in the DataFrame 
# store the outputs in a list

def extract_information():



# Your solution can differ, but your end goal is to output the properties for all products in one single list.

In [ ]:
# Calling the function


We can use these extracted properties to build a more refined shopping assistant. Try doing that on your own. It can prompt the users for their rough requirements and suggest products based on the extracted quantitative details.


Apart from Pydantic classes, you can also define JSON style schema for both Gemini and OpenAI.

In [ ]:
# Gemini

laptop_schema = {
    "type": "object",
    "properties": {
        "Brand": {"type": "string"},
        "Model Name": {"type": "string"},
        "GPU processor": {"type": "string"},
        "Display Resolution": {"type": "string"},
        "Weight": {"type": "string"},
        "Processor": {"type": "string"},
        "Clock speed": {"type": "string"},
        "Budget": {"type": "string"}
    },
    "required": [
        "Brand",
        "Model Name",
        "GPU processor",
        "Display Resolution",
        "Weight",
        "Processor",
        "Clock speed",
        "Budget"
    ]
}

response = client.models.generate_content(
    model=,
    contents=,
    config={
        "response_mime_type": "application/json",
        "response_schema": laptop_schema
    }
)

print(response.text)

In [ ]:
# OpenAI

schema = {
        "format": {
            "type": "json_schema",
            "name": "Laptop",
            "schema": {
                "type": "object",
                "properties": {
                    "Brand": {"type": "string"},
                    "Model Name": {"type": "string"},
                    "GPU processor": {"type": "string"},
                    "Display Resolution": {"type": "string"},
                    "Weight": {"type": "string"},
                    "Processor": {"type": "string"},
                    "Clock speed": {"type": "string"},
                    "Budget": {"type": "string"}
                },
                "required": [
                    "Brand",
                    "Model Name",
                    "GPU processor",
                    "Display Resolution",
                    "Weight",
                    "Processor",
                    "Clock speed",
                    "Budget"
                ],
                "additionalProperties": False
            },
            "strict": True
        }
}


response = client.responses.create(
model=,
input=,
text= schema
)

print(response.output_text)

The outputs from these will resemble JSON objects more. Try using these for data extraction.